# Investigate ssl_mel embeddings

Run with the **after** Python kernel, from the repository or notebooks directory. Uses the saved run configuration and a frozen ECAPA encoder on clean, centered waveform crops. No training augmentation is applied.

The interactive plot is self-contained HTML/SVG with embedded WAV playback (no Plotly, widgets, CDN, or server). Sample limits and class filters are configured below. The downstream task predicts Slakh instrument identity from frozen embeddings; train/validation/test tracks are disjoint **for the probes**. These tracks may have been used in SSL pretraining, so this is not a claim of generalization to audio unseen by the encoder.


In [ ]:
from pathlib import Path
import sys, json, random, copy, math, base64, io, html, uuid
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import soundfile as sf
import yaml
import lmdb
from tqdm.auto import tqdm
from IPython.display import display, HTML
from sklearn.manifold import TSNE
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

ROOT = Path.cwd().resolve()
if not (ROOT / "after").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "after").is_dir(), "Open this notebook in AFTER or AFTER/notebooks."
sys.path.insert(0, str(ROOT))
import gin
import after
from after.encoder_ssl.model import build_ssl_model
from after.dataset.audio_example import AudioExample

RUN = ROOT / "ssl_runs/ssl_mel"
CHECKPOINT = None  # None selects the numerically latest checkpoint; or set a Path
DB_PATH = Path("/fast-1/nils/datasets_processed/slakh_audio")
SLAKH_ROOT = Path("/data/nils/saved/datasets/instruments/slakh2100_flac_redux")
DEVICE = "cuda:1" if torch.cuda.is_available() else "cpu"
SEED = 42
MAX_SAMPLES = 2000
MAX_SCAN = 20000  # cap metadata reads when instrument filters are restrictive
INSTRUMENT_CLASSES = []  # e.g. ["Guitar", "Bass", "Piano"]; [] means all
INSTRUMENTS = []  # e.g. ["Acoustic Guitar (steel)"]; [] means all
LABEL = "instrument"  # or "instrument_class"
BATCH_SIZE = 8
CROP_SAMPLES = None  # None uses the saved global view length
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)


In [ ]:
gin.clear_config()
gin.parse_config_file(str(RUN / "config.gin"))
checkpoint = (max(RUN.glob("checkpoint*.pt"),
                  key=lambda p: int(p.stem.removeprefix("checkpoint")))
              if CHECKPOINT is None else Path(CHECKPOINT))
trainer = build_ssl_model(device="cpu")
state = torch.load(checkpoint, map_location="cpu", weights_only=True)
trainer.load_state_dict(state["model_state"], strict=True)
del state
trainer.eval().requires_grad_(False).to(DEVICE)
SR = int(gin.query_parameter("%SR"))
crop_samples = (int(gin.query_parameter("SSLViewCollator.n_signal_global"))
                if CROP_SAMPLES is None else int(CROP_SAMPLES))
print(f"{checkpoint.name}: clean {crop_samples / SR:.2f}s crops at {SR} Hz on {DEVICE}")


## Sample the Slakh database

Reads only the selected waveforms for embedding extraction. Older databases omit instrument metadata; labels are recovered from Slakh YAML using the stored source path. Randomized LMDB keys avoid selecting only the beginning of a partially prepared dataset. The scan stops at MAX_SAMPLES accepted examples or MAX_SCAN examined entries. It does not estimate the whole database's class distribution.


In [ ]:
# Close the notebook's previous handle before re-running this cell.
if "env" in globals():
    env.close()
env = lmdb.open(str(DB_PATH), readonly=True, lock=False, readahead=False)
track_metadata = {}

def slakh_identity(metadata):
    source = Path(metadata["path"])
    track, stem = source.parent.parent.name, source.stem
    split = source.parent.parent.parent.name
    track_key = f"{split}/{track}"
    if "instrument" in metadata and "instrument_class" in metadata:
        identity = {k: metadata[k] for k in ("instrument", "instrument_class")}
    else:
        if track_key not in track_metadata:
            with (SLAKH_ROOT / split / track / "metadata.yaml").open() as handle:
                track_metadata[track_key] = yaml.safe_load(handle)["stems"]
        info = track_metadata[track_key][stem]
        identity = dict(instrument=info["midi_program_name"],
                        instrument_class=info["inst_class"])
    # Track ID groups together every stem and chunk from a composition.
    return dict(**identity, track=track, stem=stem, split=split, path=str(source))

with env.begin() as txn:
    keys = list(txn.cursor().iternext(values=False))
rng = np.random.default_rng(SEED)
order = rng.permutation(len(keys))[:min(MAX_SCAN, len(keys))]
records = []
with env.begin() as txn:
    for i in tqdm(order, desc="Sample metadata"):
        key = keys[int(i)]
        ae = AudioExample(txn.get(key))
        metadata = ae.get_metadata()
        row = slakh_identity(metadata)
        if INSTRUMENT_CLASSES and row["instrument_class"] not in INSTRUMENT_CLASSES:
            continue
        if INSTRUMENTS and row["instrument"] not in INSTRUMENTS:
            continue
        records.append(dict(row, key=key, chunk_index=metadata["chunk_index"]))
        if len(records) >= MAX_SAMPLES:
            break
assert records, "No matching examples; revise filters or increase MAX_SCAN."
samples = pd.DataFrame(records)
print(f"Selected {len(samples)} / {len(keys)} DB entries; {samples.track.nunique()} tracks.")
display(samples.groupby(["instrument_class", "instrument"]).size().rename("sampled_chunks").to_frame())

def clean_crop(index):
    with env.begin() as txn:
        waveform = AudioExample(txn.get(samples.iloc[index]["key"])).get("waveform")
    assert waveform.ndim == 2 and waveform.shape[0] == 1
    start = max(0, (waveform.shape[-1] - crop_samples) // 2)
    waveform = waveform[:, start:start + crop_samples]
    if waveform.shape[-1] < crop_samples:
        waveform = np.pad(waveform, ((0, 0), (0, crop_samples - waveform.shape[-1])))
    return np.ascontiguousarray(waveform, dtype=np.float32)

embeddings = []
with torch.inference_mode():
    for start in tqdm(range(0, len(samples), BATCH_SIZE), desc="Clean embeddings"):
        audio = torch.from_numpy(np.stack([
            clean_crop(i) for i in range(start, min(start + BATCH_SIZE, len(samples)))
        ])).to(DEVICE)
        embeddings.append(trainer.student(trainer.data_transform(audio)).cpu().numpy())
Z = np.concatenate(embeddings)
assert Z.ndim == 2 and np.isfinite(Z).all()
print("Embedding matrix:", Z.shape)


## Clickable 2D projection

Set PLOT_MAX_SAMPLES and optional class filters below before running. t-SNE is the default; UMAP is optional if installed. Projection uses raw embeddings. Distances in this 2D display are exploratory, not the features used for classification.

The plot has an instrument selector, hover labels, zoom buttons and clickable audio. If autoplay is blocked, press Play in the player. For playback in another browser, open the saved HTML. Audio is embedded only for plotted points, so large plot limits increase output size.


In [ ]:
PLOT_MAX_SAMPLES = 300
PLOT_CLASSES = []  # broad instrument classes; [] means all sampled classes
PROJECTION = "tsne"  # "tsne" or "umap"

eligible = np.flatnonzero(samples.instrument_class.isin(PLOT_CLASSES).to_numpy()
                         if PLOT_CLASSES else np.ones(len(samples), dtype=bool))
plot_ids = np.random.default_rng(SEED).choice(
    eligible, size=min(PLOT_MAX_SAMPLES, len(eligible)), replace=False)
assert len(plot_ids) >= 4, "Select at least four points."
if PROJECTION == "tsne":
    xy = TSNE(n_components=2, perplexity=min(30, (len(plot_ids) - 1) / 3),
              init="pca", learning_rate="auto", random_state=SEED).fit_transform(Z[plot_ids])
elif PROJECTION == "umap":
    from umap import UMAP
    xy = UMAP(n_components=2, n_neighbors=min(15, len(plot_ids) - 1),
              random_state=SEED).fit_transform(Z[plot_ids])
else:
    raise ValueError(PROJECTION)

points = []
for point, index in tqdm(list(zip(xy, plot_ids)), desc="Embed plot audio"):
    row = samples.iloc[int(index)]
    buffer = io.BytesIO()
    sf.write(buffer, clean_crop(int(index))[0], SR, format="WAV", subtype="PCM_16")
    points.append(dict(x=float(point[0]), y=float(point[1]),
                       label=str(row[LABEL]),
                       title=f"{row['instrument']} | {row['track']}/{row['stem']} | chunk {row['chunk_index']}",
                       audio="data:audio/wav;base64," + base64.b64encode(buffer.getvalue()).decode()))

def audio_scatter_html(points):
    ident = "ssl_" + uuid.uuid4().hex
    # Escape '<' so labels cannot close the script element.
    payload = json.dumps(points).replace("<", "\\u003c")
    template = """<!doctype html><meta charset="utf-8">
<div id="IDENT" style="font:14px sans-serif;max-width:950px">
<label>Instrument <select></select></label>
<button class="in">Zoom +</button><button class="out">Zoom −</button><button class="reset">Reset</button>
<svg viewBox="0 0 900 550" style="width:100%;background:#fafafa;border:1px solid #ccc"></svg>
<p class="selected">Click a point to listen to its embedding crop.</p>
<audio controls style="width:100%"></audio></div>
<script>
(() => {
const root=document.getElementById("IDENT"), data=PAYLOAD;
const svg=root.querySelector("svg"), select=root.querySelector("select");
const labels=[...new Set(data.map(p=>p.label))].sort();
["All",...labels].forEach(label=>{const o=document.createElement("option");o.textContent=label;select.appendChild(o);});
const xs=data.map(p=>p.x),ys=data.map(p=>p.y);
const xmin=Math.min(...xs),xmax=Math.max(...xs),ymin=Math.min(...ys),ymax=Math.max(...ys);
data.forEach(p=>{
 const c=document.createElementNS("http://www.w3.org/2000/svg","circle");
 c.setAttribute("cx",30+840*(p.x-xmin)/(xmax-xmin||1));
 c.setAttribute("cy",520-490*(p.y-ymin)/(ymax-ymin||1));
 c.setAttribute("r",5);c.setAttribute("fill", "hsl("+(labels.indexOf(p.label)*137.508%360)+",65%,45%)");
 c.style.cursor="pointer";c.dataset.label=p.label;
 const title=document.createElementNS(svg.namespaceURI,"title");title.textContent=p.title;c.appendChild(title);
 c.onclick=()=>{root.querySelector(".selected").textContent=p.title;
 const player=root.querySelector("audio");player.src=p.audio;player.play().catch(()=>{});};
 svg.appendChild(c);
});
select.onchange=()=>svg.querySelectorAll("circle").forEach(c=>c.style.display=(select.value==="All"||c.dataset.label===select.value)?"":"none");
let scale=1;const zoom=()=>svg.setAttribute("viewBox", [450-450/scale,275-275/scale,900/scale,550/scale].join(" "));
root.querySelector(".in").onclick=()=>{scale*=1.4;zoom();};
root.querySelector(".out").onclick=()=>{scale/=1.4;zoom();};
root.querySelector(".reset").onclick=()=>{scale=1;zoom();};
})();
</script>"""
    return template.replace("IDENT", ident).replace("PAYLOAD", payload)

plot_html = audio_scatter_html(points)
display(HTML(plot_html))
# Explicit export: browser playback also works when a notebook viewer blocks JS.
plot_path = RUN / "embedding_explorer.html"
plot_path.write_text(plot_html, encoding="utf-8")
print("Standalone interactive plot:", plot_path)


## Latent distributions

Histograms use all selected embeddings before scaling. A low-variance dimension can indicate unused capacity; the effective rank summarizes how many independent directions carry variance. These statistics depend on the selected instruments and audio crop length.


In [ ]:
stats = pd.DataFrame({
    "mean": Z.mean(0), "std": Z.std(0), "min": Z.min(0), "max": Z.max(0),
    "q01": np.quantile(Z, .01, axis=0), "q99": np.quantile(Z, .99, axis=0),
})
display(stats)
BINS = 50
columns = 8
rows = math.ceil(Z.shape[1] / columns)
fig, axes = plt.subplots(rows, columns, figsize=(20, 2.1 * rows), squeeze=False)
for d, ax in enumerate(axes.flat):
    if d >= Z.shape[1]:
        ax.axis("off")
        continue
    ax.hist(Z[:, d], bins=BINS, color="steelblue", alpha=.85)
    ax.set_title(f"d{d}: std={stats.loc[d, 'std']:.3g}", fontsize=9)
fig.tight_layout()
plt.show()
singular = np.linalg.svd(Z - Z.mean(0), compute_uv=False)
variance = singular ** 2
p = variance / max(variance.sum(), np.finfo(float).eps)
effective_rank = np.exp(-np.sum(p[p > 0] * np.log(p[p > 0]))) if variance.sum() else 0.
print("Effective covariance rank:", round(effective_rank, 2), "/", Z.shape[1])
print("Dimensions with std < 1e-3:", int((stats["std"] < 1e-3).sum()))


## Downstream instrument classification

Use the original D-dimensional embeddings. Tracks are split 60/20/20; the scaler is fitted only on the training subset. k is selected using validation macro-F1. The two-layer head is Linear(D,128) → ReLU → Linear(128,C), trained with cross entropy and selected by validation macro-F1. The test set is evaluated once after selection. Classes absent from probe training are reported and excluded from closed-set scoring; increasing MAX_SAMPLES usually improves coverage. No encoder gradients are computed.


In [ ]:
indices = np.arange(len(samples))
groups = samples.track.to_numpy()
assert len(np.unique(groups)) >= 5, "Select more tracks for grouped evaluation."
trainval, test = next(GroupShuffleSplit(n_splits=1, test_size=.2, random_state=SEED).split(indices, groups=groups))
tr, va = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED + 1).split(trainval, groups=groups[trainval]))
train, val = trainval[tr], trainval[va]
assert not (set(groups[train]) & set(groups[val]) or
            set(groups[train]) & set(groups[test]) or set(groups[val]) & set(groups[test]))
labels = samples[LABEL].to_numpy()
encoder = LabelEncoder().fit(labels[train])
known = set(encoder.classes_)
for name, idx in [("validation", val), ("test", test)]:
    unseen = sorted(set(labels[idx]) - known)
    print(name, "unseen classes:", unseen,
          "excluded examples:", int(sum(v not in known for v in labels[idx])))
val = np.array([i for i in val if labels[i] in known], dtype=int)
test = np.array([i for i in test if labels[i] in known], dtype=int)
assert len(val) and len(test) and len(known) >= 2, "Insufficient closed-set coverage."
scaler = StandardScaler().fit(Z[train])
X = scaler.transform(Z).astype(np.float32)
y_train, y_val, y_test = [encoder.transform(labels[i]) for i in (train, val, test)]
print("Probe train / validation / test:", len(train), len(val), len(test))
display(pd.crosstab(samples[LABEL], pd.Series(
    np.select([np.isin(indices, train), np.isin(indices, val), np.isin(indices, test)],
              ["train", "validation", "test"], default="excluded"), name="probe_split")))

def scores(y, prediction):
    return dict(accuracy=accuracy_score(y, prediction),
                balanced_accuracy=balanced_accuracy_score(y, prediction),
                macro_f1=f1_score(y, prediction, average="macro", zero_division=0))

knn_trials = []
best_knn, best_score = None, -1.
for k in [1, 3, 5, 10, 20]:
    if k > len(train):
        continue
    candidate = KNeighborsClassifier(n_neighbors=k, metric="euclidean", weights="distance")
    candidate.fit(X[train], y_train)
    score = scores(y_val, candidate.predict(X[val]))
    knn_trials.append(dict(k=k, **score))
    if score["macro_f1"] > best_score:
        best_knn, best_score = candidate, score["macro_f1"]
display(pd.DataFrame(knn_trials))


In [ ]:
HEAD_EPOCHS = 30
HEAD_HIDDEN = 128
HEAD_BATCH_SIZE = 128
HEAD_LR = 1e-3
HEAD_DEVICE = "cpu"  # small head; GPU is optional

torch.manual_seed(SEED)
head = torch.nn.Sequential(
    torch.nn.Linear(Z.shape[1], HEAD_HIDDEN),
    torch.nn.ReLU(),
    torch.nn.Linear(HEAD_HIDDEN, len(encoder.classes_)),
).to(HEAD_DEVICE)
optimizer = torch.optim.Adam(head.parameters(), lr=HEAD_LR)
xt = torch.from_numpy(X[train]).to(HEAD_DEVICE)
yt = torch.from_numpy(y_train).to(HEAD_DEVICE)
xv = torch.from_numpy(X[val]).to(HEAD_DEVICE)
history, best_state, best_score = [], None, -1.
generator = torch.Generator().manual_seed(SEED)
for epoch in tqdm(range(HEAD_EPOCHS), desc="Two-layer probe"):
    head.train()
    total = 0.
    for ids in torch.randperm(len(train), generator=generator).split(HEAD_BATCH_SIZE):
        ids = ids.to(HEAD_DEVICE)
        optimizer.zero_grad()
        loss = torch.nn.functional.cross_entropy(head(xt[ids]), yt[ids])
        loss.backward()
        optimizer.step()
        total += loss.item() * len(ids)
    head.eval()
    with torch.no_grad():
        pred = head(xv).argmax(1).cpu().numpy()
    metrics = scores(y_val, pred)
    history.append(dict(epoch=epoch + 1, loss=total / len(train), **metrics))
    if metrics["macro_f1"] > best_score:
        best_score = metrics["macro_f1"]
        best_state = copy.deepcopy(head.state_dict())
head.load_state_dict(best_state)
pd.DataFrame(history).set_index("epoch")[["loss", "macro_f1"]].plot(subplots=True, figsize=(9, 5))
plt.show()

head.eval()
with torch.no_grad():
    head_prediction = head(torch.from_numpy(X[test]).to(HEAD_DEVICE)).argmax(1).cpu().numpy()
knn_prediction = best_knn.predict(X[test])
display(pd.DataFrame({
    f"kNN (k={best_knn.n_neighbors})": scores(y_test, knn_prediction),
    "Two-layer head": scores(y_test, head_prediction),
    "Train-majority baseline": scores(y_test, np.full_like(y_test, np.bincount(y_train).argmax())),
}).T)
for name, prediction in [("kNN", knn_prediction), ("Two-layer head", head_prediction)]:
    print(name)
    present = np.union1d(y_test, prediction)
    display(pd.DataFrame(classification_report(
        y_test, prediction, labels=present, target_names=encoder.classes_[present],
        output_dict=True, zero_division=0)).T)


The interactive HTML and notebook outputs contain audio from your local dataset. To examine another checkpoint or sampling filter, change the configuration and rerun from the model-loading cell. Closing the notebook releases the model and LMDB resources; the optional cell below releases them immediately.


In [ ]:
# Run when finished:
# env.close()
# del trainer
# if torch.cuda.is_available():
#     torch.cuda.empty_cache()
